# 09 — LangGraph Workflow

The **AdaptiveWorkflow** is the *full* state machine that ties everything together. It's built with **LangGraph** and orchestrates classification, strategy selection, retrieval, grading, rewriting, and answer generation in a single graph.

This notebook visualises the graph and walks through each node.

## 1. Import the Workflow

In [ ]:
import sys
sys.path.insert(0, "..")

try:
    from backend.adaptive_rag.langgraph_workflows.adaptive_workflow import AdaptiveWorkflow, WorkflowState
    from backend.adaptive_rag.router.query_classifier import QueryClassifier
    from backend.adaptive_rag.router.strategy_selector import StrategySelector
    from backend.adaptive_rag.router.confidence_scorer import ConfidenceScorer
    print("✓ All imports successful")
except ImportError as e:
    print(f"⚠ Backend import: {e}")
    print("→ The following cells describe the structure; run in the project root with `pip install -e .`")

## 2. The Complete Graph

```
           ┌──────────────┐
           │classify_query│
           └──────┬───────┘
                  │
                  ▼
           ┌──────────────┐
           │  route_query  │
           └──────┬───────┘
                  │
                  ▼
           ┌──────────────────┐
           │retrieve_documents │
           └──────┬───────────┘
                  │
                  ▼
           ┌──────────────┐
           │grade_documents │
           └──────┬───────┘
                  │
        ┌─────────┴─────────┐
        ▼                   ▼
   ┌──────────┐      ┌──────────────┐
   │generate  │      │rewrite_query  │
   │_answer   │      └──────┬───────┘
   └────┬─────┘             │
        │                   │
        ▼                   ▼
   ┌──────────┐      ┌──────────────┐
   │grade     │      │retrieve_docs │
   │_answer   │      │(re‑retrieve) │
   └────┬─────┘      └──────────────┘
        │
   ┌────┴────┬──────────┐
   ▼         ▼          ▼
  END   generate     web_search
        _answer
```

## 3. WorkflowState

The entire pipeline communicates through a `TypedDict` state:

In [ ]:
print("WorkflowState fields:")
for field, desc in {
    "query": "The current (possibly rewritten) query",
    "query_type": "factual / exploratory / comparative / procedural / opinion",
    "strategy": "direct_llm / document_rag / web_search_rag / hybrid_rag / self_rag",
    "documents": "List of dicts from vector retrieval",
    "web_results": "List of dicts from web search",
    "answer": "Generated answer text",
    "grade": "'relevant' or 'not_relevant'",
    "answer_quality": "'good', 'hallucination', or 'not_useful'",
    "generation_count": "Number of generation attempts",
    "web_search_count": "Number of web searches performed",
}.items():
    print(f"  {field:<20} {desc}")

## 4. Build the Workflow (Minimal Dependencies)

In [ ]:
from langchain.embeddings import FakeEmbeddings
from langchain.vectorstores import FAISS
from langchain.schema import Document

fake_emb = FakeEmbeddings(size=768)
sample_docs = [
    Document(page_content="The Eiffel Tower is in Paris, France. Built in 1889.", metadata={"source": "geo.txt"}),
    Document(page_content="Python was created by Guido van Rossum in 1991.", metadata={"source": "history.txt"}),
    Document(page_content="Climate change is primarily caused by greenhouse gas emissions.", metadata={"source": "science.txt"}),
]
vector_store = FAISS.from_documents(sample_docs, fake_emb)

print("✓ Vector store ready")

## 5. Conditional Edges Explained

The graph has three decision points:

### `_decide_route` (after classify)
Currently returns `"simple"` always. In production this could route differently based on complexity.

### `_decide_grade` (after grade_documents)
- `"relevant"` → generate answer
- `"not_relevant"` → rewrite query & re‑retrieve (up to `max_iterations`)

### `_decide_answer_quality` (after grade_answer)
- `"good"` → **END** (success)
- `"hallucination"` → re‑generate (try again)
- `"not_useful"` → web search for fresh sources

In [ ]:
print("Decision 1: grade_documents → decide_grade")
print("  relevant    → generate_answer")
print("  not_relevant → rewrite_query (up to 3 iterations)")
print()
print("Decision 2: grade_answer → decide_answer_quality")
print("  good         → END")
print("  hallucination → generate_answer (retry)")
print("  not_useful   → web_search (fetch fresh sources)")

## 6. Graders & Generators Used

| Component | Class | What it does |
|-----------|-------|-------------|
| RelevanceGrader | `graders/relevance_grader.py` | Scores document relevance to query |
| HallucinationGrader | `graders/hallucination_grader.py` | Checks if answer is grounded in context |
| AnswerGrader | `graders/answer_grader.py` | Assesses answer quality |
| QueryRewriter | `graders/query_rewriter.py` | Rewrites query for better retrieval |
| AnswerGenerator | `generator/answer_generator.py` | Generates answer from query + context |
| ContextBuilder | `generator/context_builder.py` | Joins documents into a single context string |
| ResponseFormatter | `generator/response_formatter.py` | Structures the final output |

## 7. Visualise the Graph (if langgraph is installed)

In [ ]:
try:
    from IPython.display import Image, display
    from langchain.embeddings import FakeEmbeddings
    from langchain.vectorstores import FAISS
    from langchain.schema import Document
    
    # Quick instantiation for visualisation
    class PlaceholderLLM:
        async def generate(self, prompt): return ""
        async def generate_with_context(self, q, c): return ""
        async def grade_relevance(self, d, q): return True
        async def detect_hallucination(self, a, c): return False
        async def grade_answer(self, a, c): return True
    
    llm = PlaceholderLLM()
    workflow = AdaptiveWorkflow(llm=llm, vector_store=vector_store)
    
    # Get the graph PNG (requires pygraphviz)
    try:
        graph_image = workflow.workflow.get_graph().draw_png()
        display(Image(graph_image))
    except Exception:
        print("Graph visualisation requires pygraphviz.")
        print("Install: brew install graphviz && pip install pygraphviz")
    
    print("\nGraph nodes:", list(workflow.workflow.get_graph().nodes))
    
except Exception as e:
    print(f"Could not build workflow: {e}")
    print("\n(Full pipeline requires a real LLM or deeper mock)")

## 8. Running the Workflow

When you have a real LLM configured (OpenAI / Anthropic), you can run the full pipeline:

In [ ]:
# Example (requires LLM + vector_store):
# 
# from backend.adaptive_rag.langgraph_workflows.adaptive_workflow import AdaptiveWorkflow
# 
# workflow = AdaptiveWorkflow(llm=your_llm, vector_store=your_vector_store)
# result = await workflow.run("What is the Eiffel Tower and where is it located?")
# print(result["answer"])

print("See notebook 10 for end-to-end examples with real components.")

> **Next:** [10 — Real World Examples](./10_Real_World_Examples.ipynb) — end-to-end comparisons across strategies